# code_y1：Y1 线性基线——从数据检查到提交文件

这是一份偏教学用途的 Notebook。它不会刻意压缩说明，而是尽量解释：

- 每一段代码为什么存在；
- 数据在各步骤中的形状（shape）如何变化；
- 常见 NumPy / Pandas 函数在这里具体做什么；
- 线性模型、均方误差、梯度下降和 RankIC 之间是什么关系；
- 哪些地方容易产生数据泄漏、内存压力或提交格式错误。

本文件只训练和预测 **Y1**。


## 1. 目标

最终目标是从 `data.z` 中读取数值特征，训练一个最基础的线性模型，并生成：

```text
outputs/baseline_y1.npy
```

该预测文件应满足：

- 类型为 `float32`；
- 形状为 `(测试时间点数量, 股票数量)`；
- 每一行对应一个测试时间点；
- 每一列对应一只股票；
- 文件可以被 `numpy.load(...)` 直接读取。

这份线性模型更适合用来验证完整流程，而不是直接追求最高比赛成绩。


## 2. 整体算法路线

整条流程可以概括为：

```text
data.z
  ↓ 解压与反序列化
data 字典
  ↓ 根据官方时间索引切分
训练集 / 验证集 / 测试集
  ↓ mask 与有限值过滤
有效的 (特征, Y1 标签) 样本
  ↓ 只用训练集计算均值和标准差
标准化后的 99 个数值特征
  ↓ 小批量随机梯度下降（Mini-batch SGD）
线性模型：预测值 = 特征 @ 权重 + 偏置
  ↓ 在验证集逐时间计算截面 RankIC
验证指标
  ↓ 对测试集逐时间预测
baseline_y1.npy
```

这里最重要的两个原则是：

1. **时间不能打乱切分。** 金融数据有严格的先后顺序，随机切分可能把未来信息泄漏到训练集。
2. **标准化参数只能从训练集计算。** 验证集和测试集的均值、标准差也属于未来信息。


### 2.1 关键假设与限制

- 当前模型只使用 `num_x` 中的 99 个连续数值特征。
- 暂不使用 `cat_x` 中的类别特征。
- 暂不构造历史窗口，因此每个时间点只使用该时刻的特征。
- 模型是线性的，不能自动学习复杂的非线性关系和特征交互。
- 无效预测位置暂时填入中性分数 `0.5`。如果比赛说明对无效位置有不同要求，应以官方说明为准。
- `data.z` 很大，解压后会占用更多内存；首次读取、全量重复检查和训练都可能耗时较长。


## 3. 环境与基础设置

建议从上到下依次执行，不要跳着运行。Notebook 中后面的变量通常依赖前面的单元。

需要的第三方库：

- `zstd`：解压 Zstandard 数据；
- `numpy`：数组计算、线性代数和保存 `.npy`；
- `pandas`：读取 Pickle、构造检查表和计算平均秩。

Python 标准库：

- `pickle`：把二进制序列恢复为 Python 对象；
- `pathlib.Path`：以跨平台方式管理路径。

如果导入时出现 `ModuleNotFoundError`，可以在新的 Notebook 单元中运行：

```python
%pip install numpy pandas zstd
```

安装完成后建议重启 Kernel，再从第一个代码单元开始执行。


In [1]:
# 标准库：随 Python 自带。
import pickle
from pathlib import Path

# 第三方库：需要预先安装。
import numpy as np
import pandas as pd
import zstd


### 3.1 集中设置输入、输出和随机种子

把经常修改的路径和参数集中放在前面，可以避免它们散落在 Notebook 各处。

`Path` 对象支持使用 `/` 拼接路径。例如：

```python
Path("outputs") / "baseline_y1.npy"
```

比手工拼接反斜杠或斜杠更不容易出错。


In [2]:
DATA_PATH = Path("../../data.z")
OUTPUT_DIR = Path("outputs")
PREDICTION_PATH = OUTPUT_DIR / "baseline_y1.npy"
MODEL_PATH = OUTPUT_DIR / "linear_sgd_model_y1.npz"


## 4. 读取 `data.z`

这个文件并不是“直接用 zstd 解压后就得到 NumPy 数组”，而是经过了多层封装：

1. `pandas.read_pickle(path)` 先读取最外层 Pickle；
2. 得到的内容仍是 Zstandard 压缩字节；
3. `zstd.loads(...)` 解压这些字节；
4. `pickle.loads(...)` 再把解压后的字节恢复为 Python 字典。

注意：`pickle` 文件只能从可信来源加载，因为恶意 Pickle 可能在读取时执行代码。


In [3]:
def read_zstd_pickle(path):
    """读取主办方提供的 data.z，并返回其中保存的 Python 对象。

    参数
    ----
    path : str 或 pathlib.Path
        data.z 的文件路径。

    返回
    ----
    object
        本项目中预期为一个字典，包含特征、标签、掩码和切分索引。

    内存提示
    --------
    这个函数会把完整数据载入内存。压缩文件大小不等于解压后的内存占用。
    """
    compressed_bytes = pd.read_pickle(path)
    decompressed_bytes = zstd.loads(compressed_bytes)
    return pickle.loads(decompressed_bytes)


In [4]:
# 这是本 Notebook 最可能耗时、占内存最多的单个读取步骤。
data = read_zstd_pickle(DATA_PATH)
print(f"读取完成，共有 {len(data)} 个顶层键。")


读取完成，共有 9 个顶层键。


## 5. 认识数据结构

主要数组通常采用以下维度：

- `T`：时间点数量；
- `S`：股票数量；
- `F`：连续数值特征数量，本题为 99；
- `C`：类别特征数量，本题为 9。

常见形状：

| 键 | 典型形状 | 含义 |
|---|---:|---|
| `num_x` | `(T, S, F)` | 连续数值特征 |
| `cat_x` | `(T, S, C)` | 类别特征 |
| `y1` | `(T, S)` | Y1 标签 |
| `mask_x` | `(T, S)` | 该位置的特征是否有效 |
| `mask_y` | `(T, S)` | 该位置是否属于标签范围 |

`shape` 表示数组每一维的长度，`dtype` 表示单个元素的存储类型。


In [5]:
# 用小表格展示结构，不直接打印大数组内容。
structure_rows = []
for key in sorted(data.keys()):
    value = data[key]
    structure_rows.append(
        {
            "key": key,
            "python_type": type(value).__name__,
            "shape": getattr(value, "shape", None),
            "dtype": str(getattr(value, "dtype", "")),
        }
    )

data_structure_df = pd.DataFrame(structure_rows)
display(data_structure_df)


,key,python_type,shape,dtype
0,cat_x,ndarray,"(3603, 5282, 9)",int64
1,mask_x,ndarray,"(3603, 5282)",bool
2,mask_y,ndarray,"(3603, 5282)",bool
3,num_x,ndarray,"(3603, 5282, 99)",float32
4,test_start_idx,int64,(),int64
5,train_start_idx,int64,(),int64
6,valid_start_idx,int64,(),int64
7,y1,ndarray,"(3603, 5282)",float32
8,y2,ndarray,"(3603, 5282)",float32


## 6. 按官方时间索引切分数据

Python 切片 `array[start:stop]` 包含 `start`，但不包含 `stop`，即半开区间 `[start, stop)`。

本 Notebook 使用：

- 训练集：`[train_start_idx, valid_start_idx)`；
- 验证集：`[valid_start_idx, test_start_idx)`；
- 测试集：`[test_start_idx, T)`；
- `pretrain`：训练起点之前的时间，仅用于数据概览，不用于当前模型训练。

这种按时间切分的方式模拟“用过去预测未来”。


In [6]:
# 从 num_x 的形状中读取三个核心维度。
T, STOCK_COUNT, FEATURE_COUNT = data["num_x"].shape
CATEGORY_COUNT = data["cat_x"].shape[2]

# NumPy 整数转为普通 Python int，便于 range 和 slice 使用。
train_start_idx = int(data["train_start_idx"])
valid_start_idx = int(data["valid_start_idx"])
test_start_idx = int(data["test_start_idx"])

print(
    f"T={T}, 股票数={STOCK_COUNT}, "
    f"数值特征数={FEATURE_COUNT}, 类别特征数={CATEGORY_COUNT}"
)


T=3603, 股票数=5282, 数值特征数=99, 类别特征数=9


In [7]:
# slice 对象只描述起点和终点，本身不会复制数组。
split_slices = {
    "pretrain": slice(0, train_start_idx),
    "train": slice(train_start_idx, valid_start_idx),
    "valid": slice(valid_start_idx, test_start_idx),
    "test": slice(test_start_idx, T),
}

# 当前 Y1 流程需要沿时间维切分的数组。
time_series_keys = ["num_x", "cat_x", "y1", "mask_x", "mask_y"]

# 基础切片通常返回原数组的“视图”，不会立即复制整块数据。
train_data = {
    key: data[key][split_slices["train"]]
    for key in time_series_keys
}
valid_data = {
    key: data[key][split_slices["valid"]]
    for key in time_series_keys
}
test_data = {
    key: data[key][split_slices["test"]]
    for key in time_series_keys
}


In [8]:
split_summary_rows = []
for split_name, split_slice in split_slices.items():
    split_summary_rows.append(
        {
            "split": split_name,
            "start_inclusive": split_slice.start,
            "stop_exclusive": split_slice.stop,
            "time_count": split_slice.stop - split_slice.start,
        }
    )

split_summary_df = pd.DataFrame(split_summary_rows)
display(split_summary_df)


,split,start_inclusive,stop_exclusive,time_count
0,pretrain,0,486,486
1,train,486,2918,2432
2,valid,2918,3161,243
3,test,3161,3603,442


## 7. 数据质量检查

这里区分两类“缺失”：

1. **逻辑缺失**：由 `mask_x=False` 或 `mask_y=False` 表示。数组里即使有一个数，也不应把该位置当作有效样本。
2. **物理缺失**：数组元素本身是 `NaN`、`+Inf` 或 `-Inf`。

为什么两种都要检查？因为掩码与实际数值可能不完全等价。稳妥做法是训练时同时满足：

```text
特征掩码有效
AND 标签掩码有效
AND 标签是有限数
AND 99 个特征全部是有限数
```


In [9]:
def summarize_logical_missing(split_name, split_slice):
    """统计一个时间区间内由布尔掩码声明的无效位置。"""
    feature_mask = data["mask_x"][split_slice]
    label_mask = data["mask_y"][split_slice]

    # mask.size 是布尔数组中的总元素数。
    # np.count_nonzero(mask) 等于 True 的数量。
    total_positions = int(feature_mask.size)
    valid_feature_positions = int(np.count_nonzero(feature_mask))
    valid_label_positions = int(np.count_nonzero(label_mask))

    feature_missing = total_positions - valid_feature_positions
    label_missing = total_positions - valid_label_positions
    denominator = max(total_positions, 1)

    return {
        "split": split_name,
        "total_time_stock_positions": total_positions,
        "feature_missing_positions": feature_missing,
        "feature_missing_rate": feature_missing / denominator,
        "label_missing_positions": label_missing,
        "label_missing_rate": label_missing / denominator,
    }


In [10]:
logical_missing_rows = [
    summarize_logical_missing(split_name, split_slice)
    for split_name, split_slice in split_slices.items()
]

logical_missing_df = pd.DataFrame(logical_missing_rows)
display(logical_missing_df)


,split,total_time_stock_positions,feature_missing_positions,feature_missing_rate,label_missing_positions,label_missing_rate
0,pretrain,2567052,1700684,0.662505,1843271,0.718050
1,train,12845824,5491640,0.427504,6356725,0.494848
2,valid,1283526,193924,0.151087,300554,0.234163
3,test,2334644,115486,0.049466,292106,0.125118


### 7.1 分块统计 `NaN` 和无穷值

`np.isnan`、`np.isposinf`、`np.isneginf` 会创建布尔结果。若一次对完整大数组计算，临时内存可能很大。

因此下面沿时间维分块：

```python
block = array[start:stop]
```

`time_chunk_size` 越大，循环次数越少，但峰值内存越高。


In [12]:
def count_nonfinite(array, time_chunk_size=16):
    """分块统计数组中的 NaN、正无穷和负无穷元素数量。"""
    nan_count = 0
    positive_inf_count = 0
    negative_inf_count = 0

    for start in range(0, array.shape[0], time_chunk_size):
        stop = min(start + time_chunk_size, array.shape[0])
        block = array[start:stop]

        nan_count += int(np.count_nonzero(np.isnan(block)))
        positive_inf_count += int(np.count_nonzero(np.isposinf(block)))
        negative_inf_count += int(np.count_nonzero(np.isneginf(block)))

    return nan_count, positive_inf_count, negative_inf_count


In [13]:
physical_missing_rows = []

for array_name in ["num_x", "y1"]:
    array = data[array_name]
    nan_count, positive_inf_count, negative_inf_count = count_nonfinite(array)
    total_nonfinite = nan_count + positive_inf_count + negative_inf_count

    physical_missing_rows.append(
        {
            "array": array_name,
            "total_elements": int(array.size),
            "nan_count": nan_count,
            "positive_inf_count": positive_inf_count,
            "negative_inf_count": negative_inf_count,
            "nonfinite_rate": total_nonfinite / max(int(array.size), 1),
        }
    )

physical_missing_df = pd.DataFrame(physical_missing_rows)
display(physical_missing_df)


,array,total_elements,nan_count,positive_inf_count,negative_inf_count,nonfinite_rate
0,num_x,1884073554,0,0,0,0.000000
1,y1,19031046,11558975,0,0,0.607375


### 7.2 检查同一时间截面中的完全重复特征行

对同一时间点，如果两只股票的 99 个数值特征和 9 个类别特征完全相同，就把后出现的行计为重复行。

为降低比较成本，检查分为两层：

1. 先找数值特征完全相同的候选组；
2. 再在候选组内核对类别特征。

`RUN_DUPLICATE_CHECK=False` 可以跳过这一项。完整检查可能比较耗时，但不影响后面的模型训练。


In [14]:
RUN_DUPLICATE_CHECK = True

def rows_as_byte_keys(rows):
    """把二维数组的每一行视为一个固定宽度的二进制键。

    np.unique 对一维值做去重很方便。这里先用 np.ascontiguousarray
    保证每行数据在内存中连续，再把整行 view 成一个 np.void 值，
    从而让 np.unique 可以把“整行”当作单个键比较。

    这是一种精确字节比较，不会使用浮点近似容差。
    """
    contiguous_rows = np.ascontiguousarray(rows)
    row_width_in_bytes = (
        contiguous_rows.dtype.itemsize * contiguous_rows.shape[1]
    )
    row_key_dtype = np.dtype((np.void, row_width_in_bytes))
    return contiguous_rows.view(row_key_dtype).reshape(-1)


In [15]:
def count_exact_duplicate_feature_rows(start_idx, stop_idx):
    """统计指定时间区间内完全重复的有效特征行。"""
    valid_row_count = 0
    duplicate_row_count = 0
    duplicate_group_count = 0
    time_points_with_duplicates = 0

    for time_idx in range(start_idx, stop_idx):
        valid_stocks = data["mask_x"][time_idx]
        numeric_rows = data["num_x"][time_idx, valid_stocks]
        categorical_rows = data["cat_x"][time_idx, valid_stocks]
        valid_row_count += int(numeric_rows.shape[0])

        if numeric_rows.shape[0] < 2:
            continue

        numeric_keys = rows_as_byte_keys(numeric_rows)
        _, numeric_group_ids, numeric_counts = np.unique(
            numeric_keys,
            return_inverse=True,
            return_counts=True,
        )

        time_duplicate_rows = 0
        time_duplicate_groups = 0

        # np.flatnonzero(condition) 返回 condition=True 的下标。
        for group_id in np.flatnonzero(numeric_counts > 1):
            candidate_mask = numeric_group_ids == group_id
            candidate_categories = categorical_rows[candidate_mask]
            category_keys = rows_as_byte_keys(candidate_categories)
            _, full_counts = np.unique(category_keys, return_counts=True)

            repeated_counts = full_counts[full_counts > 1]
            time_duplicate_rows += int(np.sum(repeated_counts - 1))
            time_duplicate_groups += int(repeated_counts.size)

        if time_duplicate_rows > 0:
            time_points_with_duplicates += 1
            duplicate_row_count += time_duplicate_rows
            duplicate_group_count += time_duplicate_groups

    duplicate_rate = (
        duplicate_row_count / valid_row_count
        if valid_row_count
        else 0.0
    )

    return {
        "valid_feature_rows": valid_row_count,
        "duplicate_rows_beyond_first": duplicate_row_count,
        "duplicate_groups": duplicate_group_count,
        "time_points_with_duplicates": time_points_with_duplicates,
        "duplicate_rate": duplicate_rate,
    }


In [18]:
if RUN_DUPLICATE_CHECK:
    duplicate_rows = []

    for split_name, split_slice in split_slices.items():
        result = count_exact_duplicate_feature_rows(
            split_slice.start,
            split_slice.stop,
        )
        result["split"] = split_name
        duplicate_rows.append(result)

    duplicate_df = pd.DataFrame(duplicate_rows)[
        [
            "split",
            "valid_feature_rows",
            "duplicate_rows_beyond_first",
            "duplicate_groups",
            "time_points_with_duplicates",
            "duplicate_rate",
        ]
    ]
    display(duplicate_df)
else:
    print("已跳过完全重复特征行检查。")


,split,valid_feature_rows,duplicate_rows_beyond_first,duplicate_groups,time_points_with_duplicates,duplicate_rate
0,pretrain,866368,0,0,0,0.0
1,train,7354184,0,0,0,0.0
2,valid,1089602,0,0,0,0.0
3,test,2219158,0,0,0,0.0


## 8. 线性模型原理

对一条样本，设标准化后的 99 个数值特征为向量 $x$，权重为 $w$，偏置为 $b$：

$$
\hat{y} = x^T w + b
$$

在 NumPy 中写作：

```python
predictions = features @ weights + bias
```

其中：

- `features` 形状为 `(样本数, 99)`；
- `weights` 形状为 `(99,)`；
- `features @ weights` 形状为 `(样本数,)`；
- 标量 `bias` 会通过广播加到每个样本上。

线性模型容易理解、训练快，适合作为 baseline；缺点是表达能力有限。


### 8.1 损失函数：MSE + L2 正则化

训练的目标是让预测值接近真实 Y1。当前使用均方误差：

$$
\mathrm{MSE}=\frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i-y_i)^2
$$

再加入 L2 正则项：

$$
L = \mathrm{MSE} + \lambda\lVert w\rVert_2^2
$$

- `N`：当前 batch 的样本数；
- `lambda`：代码中的 `l2`；
- L2 会惩罚过大的权重，帮助限制过拟合和数值不稳定。

注意：比赛最终指标是 RankIC，而训练损失是 MSE。两者并不完全一致，这也是 baseline 的一个限制。


In [19]:
SGD_CONFIG = {
    # epoch：完整遍历训练时间区间一次。
    "epochs": 3,

    # 每次从原始大数组中取多少个时间点。
    "time_chunk_size": 16,

    # 每次参数更新使用多少条样本。
    "batch_size": 16384,

    # 第一轮的基础学习率，后续会按 1/sqrt(epoch) 衰减。
    "learning_rate": 0.02,

    # L2 正则化强度。
    "l2": 1e-5,

    # 固定随机种子，尽量保证重复运行结果一致。
    "seed": 42,
}

TARGET_NAME = "y1"
VARIANCE_FLOOR = 1e-6

SGD_CONFIG


{'epochs': 3,
 'time_chunk_size': 16,
 'batch_size': 16384,
 'learning_rate': 0.02,
 'l2': 1e-05,
 'seed': 42}

## 9. 提取有效训练样本

原始 `num_x[start:stop]` 的形状是：

```text
(时间点数量, 股票数量, 99)
```

布尔掩码 `valid_sample_mask` 的形状是：

```text
(时间点数量, 股票数量)
```

使用：

```python
feature_block[valid_sample_mask]
```

会把前两个维度中有效的位置“拉平”为样本维，结果形状为：

```text
(有效样本数, 99)
```

这叫作 **布尔索引**。


In [20]:
def get_labeled_chunk(start_idx, stop_idx):
    """提取一个时间区间内可用于监督训练的 Y1 样本。

    有效样本必须同时满足：
    1. mask_x=True；
    2. mask_y=True；
    3. Y1 标签不是 NaN 或无穷值；
    4. 99 个数值特征全部是有限数。

    返回
    ----
    features : np.ndarray, shape (N, FEATURE_COUNT), dtype float32
    targets : np.ndarray, shape (N,), dtype float32
    """
    feature_block = data["num_x"][start_idx:stop_idx]
    target_block = data[TARGET_NAME][start_idx:stop_idx]

    finite_feature_mask = np.all(np.isfinite(feature_block), axis=2)
    valid_sample_mask = (
        data["mask_x"][start_idx:stop_idx]
        & data["mask_y"][start_idx:stop_idx]
        & np.isfinite(target_block)
        & finite_feature_mask
    )

    # copy=False 表示“能不复制就不复制”；若类型转换必须复制，NumPy 仍会复制。
    features = feature_block[valid_sample_mask].astype(
        np.float32,
        copy=False,
    )
    targets = target_block[valid_sample_mask].astype(
        np.float32,
        copy=False,
    )

    return features, targets


In [21]:
# 用训练集开头的一小块检查函数返回的形状，不打印实际大数组。
example_stop_idx = min(
    train_start_idx + SGD_CONFIG["time_chunk_size"],
    valid_start_idx,
)
example_features, example_targets = get_labeled_chunk(
    train_start_idx,
    example_stop_idx,
)

assert example_features.ndim == 2
assert example_features.shape[1] == FEATURE_COUNT
assert example_targets.ndim == 1
assert example_features.shape[0] == example_targets.shape[0]

print("示例特征形状：", example_features.shape)
print("示例标签形状：", example_targets.shape)


示例特征形状： (27966, 99)
示例标签形状： (27966,)


## 10. 只用训练集计算标准化参数

不同特征的数值尺度可能差别很大。标准化公式为：

$$
x_{standardized}=\frac{x-\mu}{\sigma}
$$

标准化的作用：

- 让不同特征处在可比较的尺度；
- 让梯度下降通常更稳定；
- 减少某些大数值特征仅因单位较大而主导梯度。

为避免一次把全部训练样本拼成超大矩阵，这里分块累加：

- $\sum x$
- $\sum x^2$
- 样本数 $N$

然后计算：

$$
\mu=\frac{\sum x}{N},\qquad
\mathrm{Var}(x)=\frac{\sum x^2}{N}-\mu^2
$$


In [22]:
# 用 float64 累加，减少大量样本求和时的舍入误差。
feature_sum = np.zeros(FEATURE_COUNT, dtype=np.float64)
feature_squared_sum = np.zeros(FEATURE_COUNT, dtype=np.float64)
standardization_sample_count = 0


In [23]:
for chunk_start in range(
    train_start_idx,
    valid_start_idx,
    SGD_CONFIG["time_chunk_size"],
):
    chunk_stop = min(
        chunk_start + SGD_CONFIG["time_chunk_size"],
        valid_start_idx,
    )
    chunk_features, _ = get_labeled_chunk(chunk_start, chunk_stop)

    if chunk_features.shape[0] == 0:
        continue

    feature_sum += chunk_features.sum(axis=0, dtype=np.float64)
    feature_squared_sum += np.square(
        chunk_features,
        dtype=np.float64,
    ).sum(axis=0)
    standardization_sample_count += chunk_features.shape[0]

assert standardization_sample_count > 0, "训练区间内没有可用样本。"
print(f"用于计算标准化参数的样本数：{standardization_sample_count:,}")


用于计算标准化参数的样本数：6,489,099


In [24]:
feature_mean = feature_sum / standardization_sample_count
feature_variance = (
    feature_squared_sum / standardization_sample_count
    - np.square(feature_mean)
)

# 理论上方差非负，但浮点误差可能产生极小负数。
# maximum 同时给极低方差特征设置下限，避免除以接近 0 的标准差。
feature_variance = np.maximum(feature_variance, VARIANCE_FLOOR)
feature_std = np.sqrt(feature_variance)

# 模型训练用 float32，降低临时内存和计算成本。
feature_mean = feature_mean.astype(np.float32)
feature_std = feature_std.astype(np.float32)

assert np.all(np.isfinite(feature_mean))
assert np.all(np.isfinite(feature_std))
assert np.all(feature_std > 0)

standardization_preview = pd.DataFrame(
    {
        "feature_index": np.arange(FEATURE_COUNT),
        "mean": feature_mean,
        "std": feature_std,
    }
)
display(standardization_preview.head(10))


,feature_index,mean,std
0,0,4.558560e-11,0.999997
1,1,3.296133e-10,0.999993
2,2,4.823264e-11,0.999996
3,3,-7.336725e-10,0.999996
4,4,2.350063e-11,0.999995
5,5,9.816029e-11,0.999996
6,6,1.902250e-10,1.000000
7,7,-1.018288e-11,1.000000
8,8,1.650752e-11,1.000000
9,9,-3.316996e-12,1.000000


## 11. 用小批量随机梯度下降训练

一次使用全部训练样本计算梯度叫 full-batch gradient descent，内存开销较大。

这里使用 **mini-batch SGD**：

1. 每个 epoch 打乱时间块顺序；
2. 读取一个时间块；
3. 标准化特征；
4. 再打乱块内样本；
5. 每次取一个 batch 计算预测、误差和梯度；
6. 立即更新权重与偏置。

对 batch 中的误差 $e=\hat{y}-y$，梯度为：

$$
\frac{\partial L}{\partial w}
=\frac{2}{N}X^Te+2\lambda w
$$

$$
\frac{\partial L}{\partial b}
=\frac{2}{N}\sum e
$$

更新规则：

$$
\theta \leftarrow \theta-\text{learning rate}\times\nabla_\theta L
$$


In [25]:
# default_rng 是 NumPy 推荐的新随机数生成接口。
rng = np.random.default_rng(SGD_CONFIG["seed"])

# 一维权重对应 99 个数值特征；偏置是一个标量。
weights = np.zeros(FEATURE_COUNT, dtype=np.float32)
bias = np.float32(0.5)

# 保存每轮训练摘要，便于观察损失是否下降。
training_history = []


In [26]:
def update_one_batch(
    batch_features,
    batch_targets,
    current_weights,
    current_bias,
    learning_rate,
):
    """完成一次前向计算、梯度计算和参数更新。"""
    batch_predictions = batch_features @ current_weights + current_bias
    prediction_error = batch_predictions - batch_targets
    batch_size = batch_features.shape[0]

    weight_gradient = (
        2.0 * batch_features.T @ prediction_error / batch_size
        + 2.0 * SGD_CONFIG["l2"] * current_weights
    )
    bias_gradient = 2.0 * prediction_error.mean()

    updated_weights = (
        current_weights - learning_rate * weight_gradient
    ).astype(np.float32)
    updated_bias = np.float32(
        current_bias - learning_rate * bias_gradient
    )

    squared_error_sum = float(np.square(prediction_error).sum())
    value_count = int(prediction_error.size)

    return (
        updated_weights,
        updated_bias,
        squared_error_sum,
        value_count,
    )


In [27]:
def train_one_epoch(epoch_idx, current_weights, current_bias):
    """遍历训练区间一次，并返回更新后的参数与本轮 MSE。"""
    learning_rate = (
        SGD_CONFIG["learning_rate"] / np.sqrt(epoch_idx + 1)
    )

    chunk_starts = np.arange(
        train_start_idx,
        valid_start_idx,
        SGD_CONFIG["time_chunk_size"],
    )
    rng.shuffle(chunk_starts)

    epoch_squared_error = 0.0
    epoch_value_count = 0

    for chunk_start in chunk_starts:
        chunk_stop = min(
            int(chunk_start) + SGD_CONFIG["time_chunk_size"],
            valid_start_idx,
        )
        chunk_features, chunk_targets = get_labeled_chunk(
            int(chunk_start),
            chunk_stop,
        )

        if chunk_features.shape[0] == 0:
            continue

        # NumPy 广播：feature_mean 和 feature_std 会沿样本维应用到每一行。
        chunk_features = (
            chunk_features - feature_mean
        ) / feature_std

        sample_order = rng.permutation(chunk_features.shape[0])

        for batch_start in range(
            0,
            chunk_features.shape[0],
            SGD_CONFIG["batch_size"],
        ):
            batch_indices = sample_order[
                batch_start:batch_start + SGD_CONFIG["batch_size"]
            ]
            batch_features = chunk_features[batch_indices]
            batch_targets = chunk_targets[batch_indices]

            (
                current_weights,
                current_bias,
                batch_squared_error,
                batch_value_count,
            ) = update_one_batch(
                batch_features,
                batch_targets,
                current_weights,
                current_bias,
                learning_rate,
            )

            epoch_squared_error += batch_squared_error
            epoch_value_count += batch_value_count

    assert epoch_value_count > 0, "本轮没有处理任何训练样本。"
    epoch_mse = epoch_squared_error / epoch_value_count

    return current_weights, current_bias, learning_rate, epoch_mse


In [28]:
for epoch_idx in range(SGD_CONFIG["epochs"]):
    weights, bias, learning_rate, epoch_mse = train_one_epoch(
        epoch_idx,
        weights,
        bias,
    )

    training_history.append(
        {
            "epoch": epoch_idx + 1,
            "learning_rate": learning_rate,
            "train_mse": epoch_mse,
        }
    )

    print(
        f"Epoch {epoch_idx + 1}/{SGD_CONFIG['epochs']} | "
        f"learning_rate={learning_rate:.6f} | "
        f"train_mse={epoch_mse:.6f}"
    )

training_history_df = pd.DataFrame(training_history)
display(training_history_df)


Epoch 1/3 | learning_rate=0.020000 | train_mse=0.083696
Epoch 2/3 | learning_rate=0.014142 | train_mse=0.082229
Epoch 3/3 | learning_rate=0.011547 | train_mse=0.082046


,epoch,learning_rate,train_mse
0,1,0.020000,0.083696
1,2,0.014142,0.082229
2,3,0.011547,0.082046


## 12. 对单个时间点生成预测

`predict_time_point(time_idx)` 会一次预测某个时间点的全部股票。

步骤：

1. 读取该时刻的 `mask_x`；
2. 再排除含 `NaN` 或无穷值的特征行；
3. 创建长度为股票数的数组，默认填 `0.5`；
4. 只对有效股票标准化并调用线性模型；
5. 把预测写回相应股票位置。

返回的一维数组始终保持固定股票顺序，形状为 `(股票数量,)`。


In [29]:
def predict_time_point(time_idx):
    """为一个时间点的全部股票生成 Y1 预测分数。"""
    feature_rows = data["num_x"][time_idx]

    valid_positions = (
        data["mask_x"][time_idx]
        & np.all(np.isfinite(feature_rows), axis=1)
    )

    # np.full 创建指定形状，并用同一个值初始化。
    predictions = np.full(
        STOCK_COUNT,
        0.5,
        dtype=np.float32,
    )

    if np.any(valid_positions):
        standardized_features = (
            feature_rows[valid_positions] - feature_mean
        ) / feature_std
        predictions[valid_positions] = (
            standardized_features @ weights + bias
        )

    return predictions


## 13. 验证指标：逐时间截面的 RankIC

RankIC 在这里等价于预测排名与真实标签排名之间的 Spearman 相关系数。

对每个验证时间点分别计算：

1. 取该时刻所有有效股票；
2. 把预测值转为排名；
3. 把真实 Y1 转为排名；
4. 计算两组排名的 Pearson 相关系数；
5. 最后对所有有效时间点的 RankIC 求平均。

为什么要逐时间计算？比赛关心的是每个时间截面内股票相对排序的质量，而不是把所有日期混在一起计算一个相关系数。

`method="average"` 表示并列值使用平均名次。


In [30]:
def rank_values(values):
    """把一维数值转换为平均秩，并返回 float64 NumPy 数组。"""
    return (
        pd.Series(values)
        .rank(method="average")
        .to_numpy(dtype=np.float64)
    )


In [31]:
def calculate_rank_ic(predictions, labels, valid_mask):
    """计算单个时间点横截面上的 Spearman RankIC。"""
    usable_mask = (
        valid_mask
        & np.isfinite(labels)
        & np.isfinite(predictions)
    )

    # 少于两个有效点时无法计算相关系数。
    if np.count_nonzero(usable_mask) < 2:
        return np.nan

    prediction_ranks = rank_values(predictions[usable_mask])
    label_ranks = rank_values(labels[usable_mask])

    # corrcoef 返回 2×2 相关系数矩阵，[0, 1] 是两组排名的相关系数。
    correlation_matrix = np.corrcoef(
        prediction_ranks,
        label_ranks,
    )
    return float(correlation_matrix[0, 1])


In [32]:
validation_rank_ic_values = []

# 严格遍历官方验证区间 [valid_start_idx, test_start_idx)。
for time_idx in range(valid_start_idx, test_start_idx):
    time_predictions = predict_time_point(time_idx)
    validation_sample_mask = (
        data["mask_y"][time_idx]
        & data["mask_x"][time_idx]
        & np.all(
            np.isfinite(data["num_x"][time_idx]),
            axis=1,
        )
    )
    time_rank_ic = calculate_rank_ic(
        time_predictions,
        data[TARGET_NAME][time_idx],
        validation_sample_mask,
    )
    validation_rank_ic_values.append(time_rank_ic)

validation_rank_ic_values = np.asarray(
    validation_rank_ic_values,
    dtype=np.float64,
)


In [33]:
valid_metric_mask = np.isfinite(validation_rank_ic_values)
valid_metric_count = int(np.count_nonzero(valid_metric_mask))

assert valid_metric_count > 0, "验证区间内没有可计算的 RankIC。"

validation_mean_rank_ic = float(
    np.mean(validation_rank_ic_values[valid_metric_mask])
)
validation_std_rank_ic = float(
    np.std(validation_rank_ic_values[valid_metric_mask])
)

validation_summary = pd.Series(
    {
        "valid_time_points": valid_metric_count,
        "mean_rank_ic": validation_mean_rank_ic,
        "std_rank_ic": validation_std_rank_ic,
        "min_rank_ic": float(
            np.min(validation_rank_ic_values[valid_metric_mask])
        ),
        "max_rank_ic": float(
            np.max(validation_rank_ic_values[valid_metric_mask])
        ),
    },
    name="Y1 validation",
)
display(validation_summary)


valid_time_points    243.000000
mean_rank_ic           0.089678
std_rank_ic            0.098659
min_rank_ic           -0.205431
max_rank_ic            0.362268
Name: Y1 validation, dtype: float64

## 14. 生成测试集预测与提交文件

测试集没有可用于训练或验证的真实标签。下面只使用训练好的模型逐时间生成预测。

最终二维数组：

```text
test_predictions.shape
== (T - test_start_idx, STOCK_COUNT)
```

`output_time_idx` 是输出数组从 0 开始的行号，`time_idx` 是原始完整数据中的时间下标。


In [34]:
# parents=True：必要时创建父目录；exist_ok=True：目录已存在也不报错。
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

test_time_count = T - test_start_idx
test_predictions = np.full(
    (test_time_count, STOCK_COUNT),
    0.5,
    dtype=np.float32,
)

print("待生成的测试预测形状：", test_predictions.shape)


待生成的测试预测形状： (442, 5282)


In [35]:
for output_time_idx, time_idx in enumerate(
    range(test_start_idx, T)
):
    test_predictions[output_time_idx] = predict_time_point(time_idx)

assert np.all(np.isfinite(test_predictions))
print("测试集预测完成。")


测试集预测完成。


In [36]:
# np.save 保存单个 NumPy 数组，文件扩展名为 .npy。
np.save(PREDICTION_PATH, test_predictions)

print(f"已保存预测文件：{PREDICTION_PATH.resolve()}")


已保存预测文件：D:\google_dl\book\友安杯\baseline_outputs\baseline_y1.npy


In [37]:
# np.savez 可以把多个命名数组保存到同一个 .npz 文件。
# 模型文件用于复现或分析，不是比赛提交文件。
np.savez(
    MODEL_PATH,
    weights=weights,
    bias=np.asarray(bias, dtype=np.float32),
    feature_mean=feature_mean,
    feature_std=feature_std,
    validation_mean_rank_ic=np.asarray(
        validation_mean_rank_ic,
        dtype=np.float64,
    ),
)

print(f"已保存模型参数：{MODEL_PATH.resolve()}")


已保存模型参数：D:\google_dl\book\友安杯\baseline_outputs\linear_sgd_model_y1.npz


## 15. 提交前检查

保存后重新读取文件，而不是只检查内存中的变量。这样可以确认：

- 实际写入磁盘的文件存在；
- `np.load` 可以正常读取；
- shape 与 dtype 符合要求；
- 文件中没有 `NaN` 或无穷值。


In [38]:
loaded_predictions = np.load(PREDICTION_PATH)

expected_shape = (test_time_count, STOCK_COUNT)
assert loaded_predictions.shape == expected_shape, (
    f"预测形状错误：实际 {loaded_predictions.shape}，"
    f"预期 {expected_shape}"
)
assert loaded_predictions.dtype == np.float32, (
    f"预测 dtype 错误：实际 {loaded_predictions.dtype}，"
    "预期 float32"
)
assert np.all(np.isfinite(loaded_predictions)), "预测中存在 NaN 或无穷值。"

output_summary = pd.Series(
    {
        "path": str(PREDICTION_PATH.resolve()),
        "shape": str(loaded_predictions.shape),
        "dtype": str(loaded_predictions.dtype),
        "min": float(loaded_predictions.min()),
        "max": float(loaded_predictions.max()),
        "mean": float(loaded_predictions.mean()),
        "file_size_mb": PREDICTION_PATH.stat().st_size / 1024**2,
    },
    name="submission check",
)
display(output_summary)


path            D:\google_dl\book\友安杯\baseline_outputs\baselin...
shape                                                 (442, 5282)
dtype                                                     float32
min                                                     -0.921947
max                                                      1.865528
mean                                                     0.515306
file_size_mb                                             8.906082
Name: submission check, dtype: object

## 16. 如何理解结果

运行结束后优先看三处：

1. `training_history_df`：训练 MSE 是否总体下降；
2. `validation_summary`：验证集平均 RankIC 及不同时间点的波动；
3. `output_summary`：提交文件的形状、类型和有限值检查是否通过。

如果训练 MSE 下降但验证 RankIC 不提升，可能原因包括：

- MSE 与 RankIC 优化目标不完全一致；
- 线性模型表达能力不足；
- 类别特征没有使用；
- 没有使用历史窗口；
- 特征与标签关系随时间变化；
- 超参数不合适或存在过拟合。


## 17. 后续改进方向

可以按由易到难的顺序尝试：

1. 调整 `learning_rate`、`epochs`、`batch_size` 和 `l2`；
2. 保存每个 epoch 的验证 RankIC，观察是否过拟合；
3. 对预测值做逐时间截面的排名或截尾；
4. 加入类别特征编码；
5. 构造过去若干时间点的滚动统计特征；
6. 尝试树模型或神经网络等非线性模型；
7. 使用更贴近排序目标的损失函数；
8. 对时间稳定性、行业暴露和不同市场阶段做分组诊断。

修改模型时仍应保持：

- 时间切分不变；
- 标准化和特征工程只使用训练期可见信息；
- 验证指标逐时间计算；
- 最终文件 shape、dtype 与顺序不变。


## 18. 常用术语与函数速查

| 名称 | 在本 Notebook 中的含义 |
|---|---|
| `shape` | 数组各维长度，例如 `(T, S, 99)` |
| `dtype` | 元素存储类型，例如 `float32` |
| `axis=0` | 沿样本维聚合，保留特征维 |
| `axis=1` | 对二维数组逐行操作 |
| `axis=2` | 对三维特征数组的最后一维操作 |
| 布尔索引 | 使用 True/False 数组筛选元素或行 |
| `np.isfinite` | 判断元素既不是 NaN，也不是正负无穷 |
| `np.all(..., axis=2)` | 要求一条样本的全部特征都满足条件 |
| `np.count_nonzero` | 统计非零或 True 元素数量 |
| `np.arange` | 生成等间隔整数序列 |
| `np.asarray` | 把输入转换为 NumPy 数组 |
| `astype` | 转换数组 dtype |
| `@` | 矩阵乘法或矩阵与向量乘法 |
| 广播 | NumPy 自动扩展较小数组以配合逐元素运算 |
| epoch | 完整遍历训练数据一次 |
| batch | 一次梯度更新使用的一小组样本 |
| learning rate | 每次沿负梯度方向更新的步长 |
| L2 正则化 | 惩罚过大的权重 |
| 数据泄漏 | 训练时使用了本不该看到的未来或验证信息 |
| RankIC | 预测排名与标签排名的相关程度 |
| `np.save` / `np.load` | 保存和读取单个 `.npy` 数组 |
| `np.savez` | 把多个数组保存到一个 `.npz` 文件 |

这份 Notebook 故意保留较多解释，后续可以根据汇报对象和自己的熟悉程度删减。
